# DBpedia SPARQL notebook

Run SPARQL queries against the [DBpedia SPARQL endpoint](https://dbpedia.org/sparql) and show results with Polars.

| | |
|---|---|
| **Endpoint** | `https://dbpedia.org/sparql` |
| **UI** | https://dbpedia.org/sparql |

In [ ]:
from typing import Any

import polars as pl
from IPython.display import display
from SPARQLWrapper import JSON, SPARQLWrapper

DBPEDIA_ENDPOINT = "https://dbpedia.org/sparql"

pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_cols(-1)


def run_sparql(query: str, endpoint: str = DBPEDIA_ENDPOINT) -> pl.DataFrame:
    """Execute a SPARQL SELECT query and return bindings as a Polars DataFrame."""
    client = SPARQLWrapper(endpoint)
    client.setQuery(query)
    client.setReturnFormat(JSON)

    payload: dict[str, Any] = client.query().convert()
    bindings = payload.get("results", {}).get("bindings", [])
    rows = [{key: value.get("value") for key, value in row.items()} for row in bindings]
    return pl.DataFrame(rows) if rows else pl.DataFrame()


def show_df(df: pl.DataFrame) -> pl.DataFrame:
    """Print the full table without truncating long cell values."""
    display(df)
    return df


def shorten_url_columns(df: pl.DataFrame) -> pl.DataFrame:
    """Replace full http URLs with the last path segment in each string column."""
    if df.is_empty():
        return df

    shortened = df
    for column in df.columns:
        if df[column].dtype != pl.String:
            continue
        if not df[column].str.starts_with("http").any():
            continue

        shortened = shortened.with_columns(
            pl.when(pl.col(column).str.starts_with("http"))
            .then(pl.col(column).str.split("/").list.last())
            .otherwise(pl.col(column))
            .alias(column)
        )

    return shortened


def show_last_part_of_url(df: pl.DataFrame) -> pl.DataFrame:
    """Shorten URL columns and display the table."""
    return shorten_url_columns(df)


Give me 10 distinct creator names. <br>
Why query is executed on that way? <br>
Without distinct we will get many results with same creator name, because some movie name could be labeled with sufix Eng,De...

In [70]:
SAMPLE_QUERY = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT DISTINCT ?creator_name WHERE {
  ?movie dbo:creator ?creator_name
}
LIMIT 10
"""

data_frame = run_sparql(SAMPLE_QUERY) 
show_last_part_of_url(data_frame)

creator_name
str
"""Chris_Coelen"""
"""Enrique_Cruz_(journalist)"""
"""J_Stevens"""
"""Karen_Knox"""
"""Mahar_(TV_channel)"""
"""Rohit–Sasi"""
"""Tiffany_Barbuzano"""
"""Aniruddha_Rajderkar"""
"""Backrooms_(web_series)"""


Query: Give me all distinct movies and

In [71]:
SAMPLE_QUERY_TWO = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT DISTINCT ?movie ?creator_name WHERE {
  ?movie dbo:creator ?creator_name . 
  ?creator_name rdfs:label ?creator_name_label .
  FILTER (STRSTARTS(LCASE(?creator_name_label), "a"))
}
LIMIT 10
"""

show_last_part_of_url(run_sparql(SAMPLE_QUERY_TWO))

movie,creator_name
str,str
"""Sandman_(DC_Comics)""","""Allen_Bert_Christman"""
"""Fairy_Godmother_(Shrek)""","""Andrew_Adamson"""
"""Work_It_(TV_series)""","""Andrew_Reich"""
"""Pepper_Dennis""","""Aaron_Harberts"""
"""Philippa_Georgiou""","""Aaron_Harberts"""
"""Paul_Stamets_(Star_Trek)""","""Aaron_Harberts"""
"""Hugh_Culber""","""Aaron_Harberts"""
"""Archer_(2009_TV_series)""","""Adam_Reed"""
"""Lana_Kane""","""Adam_Reed"""


Give me distinct pairs of publication and publisher where publication is video game

In [131]:
SAMPLE_QUERY_THREE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT DISTINCT ?publication ?publisher WHERE {
  ?publication dbo:publisher ?publisher;
              ?publication_type dbo:VideoGame .
}
LIMIT 5
"""

show_last_part_of_url(run_sparql(SAMPLE_QUERY_THREE))

publication,publisher
str,str
"""Adidas_miCoach__Adidas_miCoach__1""","""505_Games"""
"""Ananta_(video_game)""","""NetEase"""
"""Arashi:_Castles_of_Sin""","""Skydance_Media"""
"""Arco_(video_game)""","""Panic_Inc."""
"""Asgard's_Wrath_2""","""Reality_Labs"""


Give me publication which starts which publisher name starts with "s" and where publisher name is on english

In [178]:
SAMPLE_QUERY_FOUR = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT DISTINCT ?publication ?publisher_name WHERE {
  ?publication dbo:publisher ?publisher. 
  ?publisher rdfs:label ?publisher_name .
  FILTER (lang(?publisher_name) = "en")
  FILTER (STRSTARTS(LCASE(?publisher_name), "s"))
}
LIMIT 5
"""

show_last_part_of_url(run_sparql(SAMPLE_QUERY_FOUR))

publication,publisher_name
str,str
"""Fiol's_Octoechos""","""Schweipolt Fiol"""
"""The_Transmitter""","""Simons Foundation"""
"""Quanta_Magazine""","""Simons Foundation"""
"""Krytyka_Polityczna""","""Stanisław Brzozowski (philosopher)"""
"""Newsline_(magazine)""","""Style 360"""


Give me publisher and publication which starts on letter a and which publication is video game

In [110]:
SAMPLE_QUERY_FIVE = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX res: <http://dbpedia.org/resource/classes#>

SELECT DISTINCT ?publication ?publisher_name WHERE {
  ?publication ?publication_type dbo:VideoGame ;
               dbo:publisher ?publisher .
  ?publisher rdfs:label ?publisher_name .
  FILTER (lang(?publisher_name) = "en")
  FILTER (STRSTARTS(LCASE(?publisher_name), "a"))
}
LIMIT 5
"""

show_last_part_of_url(run_sparql(SAMPLE_QUERY_FIVE))

publication,publisher_name
str,str
"""Jenny_of_the_Prairie""","""Addison-Wesley"""
"""Second_Extinction""","""Avalanche Studios Group"""
"""Snakebird_(video_game)""","""Astra Logical"""
"""Bejeweled_2""","""Android (operating system)"""
"""Countdown_(video_game)""","""Access Software"""
